## Importing Libraries

In [ ]:
import os
import random
import torch
import torch.nn as nn
from transformers import DistilBertTokenizer, DistilBertModel
from torchvision import models
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

## Selecting device (CPU or GPU)
If GPU is not available then use CPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Define classes

In [ ]:
selected_classes = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

## Create word pools per class

Domain-specific words, acts like semantic representation

In [ ]:
class_word_pools = {
    'buildings': ["tall", "urban", "building", "architecture", "city",
                  "construction", "structure", "skyscraper", "wall", "concrete"],
    'forest'   : ["dense", "green", "trees", "forest", "nature",
                  "outdoor", "woodland", "leaves", "jungle", "wild"],
    'glacier'  : ["ice", "cold", "frozen", "glacier", "snow",
                  "arctic", "blue", "mountain", "frost", "icy"],
    'mountain' : ["rocky", "mountain", "peak", "landscape", "outdoor",
                  "summit", "cliff", "highland", "steep", "altitude"],
    'sea'      : ["ocean", "water", "waves", "beach", "coastal",
                  "blue", "shore", "marine", "horizon", "sea"],
    'street'   : ["road", "street", "urban", "traffic", "city",
                  "pavement", "sidewalk", "lane", "intersection", "pedestrian"]
}

## Add shared ambiguous words
Makes text less perfect, simulates real-world noise

In [ ]:
shared_words = ["outdoor", "light", "dark", "wide", "narrow",
                "large", "small", "natural", "open", "view",
                "landscape", "environment", "scene", "area", "place"]

## Generate mixed descriptions

For each class:
- 3 words from correct class
- 1 word from another class
- 1 ambiguous word

In [ ]:
def generate_random_description(cls, num_words=5):
    # Pick words from own class
    own_words    = random.sample(class_word_pools[cls], 3)

    # ✅ Pick 1-2 words from OTHER classes
    other_classes = [c for c in selected_classes if c != cls]
    other_class   = random.choice(other_classes)
    other_words   = random.sample(class_word_pools[other_class], 1)

    # ✅ Pick 1 shared ambiguous word
    shared        = random.sample(shared_words, 1)

    # Combine and shuffle
    all_words = own_words + other_words + shared
    random.shuffle(all_words)
    return " ".join(all_words)

## Test — verify mixed words

In [ ]:


print("Testing descriptions (notice mixed words):")
for cls in selected_classes:
    d1 = generate_random_description(cls)
    d2 = generate_random_description(cls)
    print(f"{cls:12} → '{d1}'")
    print(f"{'':12}   '{d2}'")
    print()

## Load transformer model

In [ ]:
tokenizer  = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')

## Move model to GPU

In [ ]:
bert_model = bert_model.to(device)
bert_model.eval()

## Freeze the model

We are not training BERT just using as feature extractor

In [ ]:
for param in bert_model.parameters():
    param.requires_grad = False

## Text sentence -> Vector

In [ ]:
def get_text_features(text):
    # convert words → numbers
    tokens = tokenizer(
        text,
        return_tensors = 'pt',
        padding        = True,
        truncation     = True,
        max_length     = 32
    ).to(device)
    with torch.no_grad():
        # Get embeddings - Produces vector representation of text
        output = bert_model(**tokens)
    return output.last_hidden_state[:, 0, :]    # return text feature vector

print("DistilBERT loaded and frozen!")
print(f"Text feature size: {get_text_features('test').shape}")

output = bert_model(**tokens) returns a tensor like: (batch_size, sequence_length, hidden_size)

In our case (1, 32, 768),
Meaning:
- 1 → number of sentences
- 32 → number of tokens (words after padding/truncation)
- 768 → features per token

output.last_hidden_state[:, 0, :]
- : → all sentences
- 0 → first token (Special token added by BERT, Represents the whole sentence meaning)
- : → all features (768 values)

## Image + Text Fusion

In [ ]:
class ImageTextFusionModel(nn.Module):
    def __init__(self, num_classes=6):
        super(ImageTextFusionModel, self).__init__()

        # Load ResNet
        resnet = models.resnet50(pretrained=True)

        # Unfreeze layer4 only
        for name, param in resnet.named_parameters():
            param.requires_grad = 'layer4' in name

        # Remove final Fully Connected layer
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])     # Output: (batch, 2048, 1, 1)

        # Project visual 2048 → 512
        self.visual_projector = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Text projector
        # Project text 768 → 512
        self.text_projector = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # --- Fusion Layer ---
        # Visual (512) + Text (512) = 1024
        # 1024 → 256 → 6 classes
        self.fusion = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    # Input: image + text_features
    # Output: class prediction (6 classes)
    def forward(self, image, text_feat):
        # Visual path (image → features)
        cnn_out     = self.cnn(image)           # Image goes through ResNet (without final layer) output - batch, 2048, 1, 1
        cnn_out     = cnn_out.view(cnn_out.size(0), -1)   # Flatten it (batch, 2048)
        visual_feat = self.visual_projector(cnn_out)      # Project to 512, Converts: 2048 → 512, Final image feature: (batch, 512)

        # Text path
        if text_feat.dim() == 3:
            text_feat = text_feat.squeeze(1)
        # Handles shape issue
        # Sometimes text comes as: (batch, 1, 768)
        # This converts it to: (batch, 768)
        text_feat = self.text_projector(text_feat)        # Converts: 768 → 512, Final text feature: (batch, 512)

        # Fusion
        fused = torch.cat([visual_feat, text_feat], dim=1)  
        # Combine both: 512 (image) + 512 (text) = 1024
        # Output: (batch, 1024)
        out   = self.fusion(fused)           # Converts: 1024 → 256 → 6 classes, Output: (batch, 6)
        # These are logits (raw scores)
        return out

# Initialize
image_text_model = ImageTextFusionModel(num_classes=6).to(device)
print("Image+Text Fusion Model ready!")
print(f"Trainable params: {sum(p.numel() for p in image_text_model.parameters() if p.requires_grad):,}")

In [ ]:
class ImageTextDataset(Dataset):
    def __init__(self, samples, selected_classes,
                 transform=None, training=False):
        self.samples          = samples
        self.selected_classes = selected_classes
        self.transform        = transform
        self.training         = training

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        cls_name        = self.selected_classes[label]

        # Load image
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # Generate text description
        if self.training:
            num_words = random.randint(4, 6)
            desc      = generate_random_description(cls_name, num_words)
        else:
            random.seed(idx)
            desc = generate_random_description(cls_name, 5)
            random.seed()

        text_feat = get_text_features(desc).squeeze(0)

        return image, text_feat, label

# Transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Build samples
train_path   = '/content/intel_data/seg_train/seg_train'
val_path     = '/content/intel_data/seg_test/seg_test'
class_to_idx = {cls: idx for idx, cls in enumerate(selected_classes)}

def build_samples(root_path, class_to_idx):
    samples = []
    for cls, idx in class_to_idx.items():
        cls_path = os.path.join(root_path, cls)
        if not os.path.exists(cls_path):
            print(f"Missing: {cls}")
            continue
        for img_file in os.listdir(cls_path):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                samples.append((os.path.join(cls_path, img_file), idx))
    return samples

train_samples = build_samples(train_path, class_to_idx)
val_samples   = build_samples(val_path,   class_to_idx)

print(f"Train samples: {len(train_samples)}")
print(f"Val samples  : {len(val_samples)}")

# Datasets
train_dataset = ImageTextDataset(train_samples, selected_classes,
                                  train_transforms, training=True)
val_dataset   = ImageTextDataset(val_samples,   selected_classes,
                                  val_transforms,   training=False)

# ✅ num_workers=0 because of dynamic BERT calls
train_loader = DataLoader(train_dataset, batch_size=32,
                          shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32,
                          shuffle=False, num_workers=0)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")